## 1. Set keywords to search for

Translate english keywords into Japanese to search for in the manifestos

Deflation = デフレーション, デフレ \
Nuclear Power = 原子力 \
China = 中国 \
Constitution = 憲法 \
Migration = 移民 (Uncommon), 外国人労働者 ("Foreign Worker", More Common)

In [32]:
import pandas as pd
import spacy
from collections import Counter

nlp = spacy.load("ja_core_news_sm")

# Make sure text length isnt too long for spacy
def chunk_text(text, max_bytes=40000):
    encoded = text.encode("utf-8")
    chunks = []
    while encoded:
        chunk = encoded[:max_bytes]
        # avoid splitting mid-character
        chunk = chunk.decode("utf-8", errors="ignore")
        chunks.append(chunk)
        encoded = encoded[len(chunk.encode("utf-8")):]
    return chunks

# Exclude common particles etc.
excluded_words = {"の", "に", "は", "を", "た", "が", "で", "て", "と", "し", "ます", "する", "な", "や", "も", "へ"}

# Tokenize manifesto text
def tokenize(csv_path):
    df = pd.read_csv(csv_path)
    all_tokens = []
    
    for text in df["text"]:
        for chunk in chunk_text(str(text)):
            doc = nlp(chunk)
            tokens = [token.text for token in doc 
                      if not token.is_space 
                      and not token.is_punct
                      and token.text not in excluded_words]
            all_tokens.extend(tokens)
    
    token_df = pd.DataFrame(all_tokens, columns=["word"])
    word_counts_df = token_df["word"].value_counts().reset_index()
    word_counts_df.columns = ["word", "count"]
    
    return word_counts_df

word_counts_df = tokenize("../../project/data/clean/Japan_sorted_manifestos.csv")

In [33]:
word_counts_df.head(10)

,word,count
0,的,566
1,化,530
2,者,444
3,等,409
4,法,370
5,支援,364
6,が,350
7,ため,331
8,で,314
9,など,313


In [34]:
# Search for count of a specific word
def wordsearch(keyword):
    count = word_counts_df.loc[word_counts_df['word'] == keyword, 'count'].item()
    return count

wordsearch('的')

566